# Train a healthy / unhealthy leaf detector

Run every cell top to bottom. Takes about **30–40 minutes** on Colab's free GPU.

At the end you download two files:

- `model_int8.tflite` — about 6 MB, the file the Android app uses
- `manifest.json` — the class names, read by the app

**First:** `Runtime → Change runtime type → T4 GPU`, then `Save`.


## 1. Check we have a GPU

In [ ]:
!nvidia-smi
# If this says "command not found", go to Runtime -> Change runtime type -> T4 GPU.

## 2. Install YOLO

In [ ]:
!pip install -q ultralytics==8.3.0
import ultralytics; ultralytics.checks()

## 3. Get the photos

We use **PlantVillage** — about 54,000 photos of leaves, sorted into folders
like `Tomato___healthy` and `Tomato___Late_blight`. It is free and public.

Run the cell. Kaggle will ask you to log in the first time (a free account is enough).

> If the download fails because the dataset moved, search Kaggle for
> *PlantVillage*, copy the new `owner/dataset-name` from the URL, and paste it
> into `SLUG` below.

In [ ]:
import kagglehub, os

SLUG = "abdallahalidev/plantvillage-dataset"     # <- change here if it moved

path = kagglehub.dataset_download(SLUG)
print("downloaded to:", path)

# Find the folder that actually holds the class sub-folders.
SRC = path
for root, dirs, files in os.walk(path):
    if any("healthy" in d.lower() for d in dirs):
        SRC = root
        break
print("using source folder:", SRC)
print("example classes:", sorted(os.listdir(SRC))[:6])

### Not using Kaggle?

Skip the cell above, upload your own zip instead, and set `SRC` to the folder
inside it. Any folder layout works as long as the sub-folder names contain the
word *healthy* for the healthy ones.

In [ ]:
# from google.colab import files
# up = files.upload()
# !unzip -q <your-file>.zip -d raw
# SRC = "raw" 

## 4. Turn it into a 2-class detection dataset

PlantVillage is a *classification* dataset — no boxes. Each photo is one leaf
filling most of the frame, so we generate a box covering the middle 90%.

Folder name contains "healthy" → **healthy**. Everything else → **unhealthy**.
We take an equal number of each so the model can't just guess.

In [ ]:
import random, shutil, pathlib

PER_CLASS  = 1500      # per class. 1500 is plenty for a demo; raise for a better model
VAL_SPLIT  = 0.2
BOX        = 0.90
IMG_TYPES  = {".jpg", ".jpeg", ".png", ".bmp"}
random.seed(42)

src = pathlib.Path(SRC)
buckets = {"healthy": [], "unhealthy": []}

for folder in sorted(p for p in src.rglob("*") if p.is_dir()):
    photos = [p for p in folder.iterdir() if p.suffix.lower() in IMG_TYPES]
    if not photos:
        continue
    label = "healthy" if "healthy" in folder.name.lower() else "unhealthy"
    buckets[label] += photos

print(f"found  {len(buckets['healthy']):>6} healthy   {len(buckets['unhealthy']):>6} unhealthy")

take = min(PER_CLASS, len(buckets["healthy"]), len(buckets["unhealthy"]))
print(f"using  {take} of each")

DATA = pathlib.Path("data")
for split in ("train", "val"):
    for kind in ("images", "labels"):
        d = DATA / kind / split
        shutil.rmtree(d, ignore_errors=True)
        d.mkdir(parents=True, exist_ok=True)

box_line = f"0.500000 0.500000 {BOX:.6f} {BOX:.6f}"
for cls_id, label in enumerate(("healthy", "unhealthy")):
    pool = buckets[label][:]
    random.shuffle(pool)
    pool = pool[:take]
    n_val = int(len(pool) * VAL_SPLIT)
    for i, photo in enumerate(pool):
        split = "val" if i < n_val else "train"
        stem = f"{label}_{i:05d}"
        shutil.copy2(photo, DATA / "images" / split / (stem + photo.suffix.lower()))
        (DATA / "labels" / split / (stem + ".txt")).write_text(f"{cls_id} {box_line}\n")

pathlib.Path("data.yaml").write_text("""path: data
train: images/train
val: images/val
names:
  0: healthy
  1: unhealthy
""")
pathlib.Path("data/classes.txt").write_text("healthy\nunhealthy\n")

for split in ("train", "val"):
    print(split, len(list((DATA / "images" / split).iterdir())), "photos")

### Have a look at what we just built

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
import random, pathlib

files = random.sample(sorted((pathlib.Path("data/images/train")).iterdir()), 6)
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, f in zip(axes.ravel(), files):
    im = Image.open(f); w, h = im.size
    ax.imshow(im)
    ax.add_patch(patches.Rectangle((w*0.05, h*0.05), w*0.90, h*0.90,
                                   fill=False, edgecolor="#97BC62", linewidth=3))
    ax.set_title(f.name.split("_")[0], color="#2C5F2D")
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Train

`yolov8n` is the nano model — the small one that fits on a phone. Always start here.

30 epochs is enough for a clear demo. Raise it to 100 for a better model.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="data.yaml",
    epochs=30,
    imgsz=640,
    batch=32,
    patience=10,
    project="runs", name="train", exist_ok=True,

    # make it cope with real photos, not just tidy ones
    hsv_s=0.7, hsv_v=0.4,     # colour and brightness
    degrees=10, scale=0.5,    # angle and distance
    fliplr=0.5, mosaic=1.0,
)

## 6. How well did it do?

In [ ]:
from IPython.display import Image as Show, display

metrics = model.val(data="data.yaml")
print(f"\noverall accuracy (mAP50-95): {metrics.box.map*100:.1f}")
print(f"overall accuracy (mAP50):    {metrics.box.map50*100:.1f}\n")
for i, name in enumerate(["healthy", "unhealthy"]):
    print(f"  {name:<12}{metrics.box.maps[i]*100:.1f}")

display(Show("runs/train/results.png", width=900))
display(Show("runs/train/confusion_matrix_normalized.png", width=520))

### Try it on a few photos

In [ ]:
import pathlib, random
from IPython.display import Image as Show, display

samples = random.sample(sorted(pathlib.Path("data/images/val").iterdir()), 4)
results = model.predict([str(p) for p in samples], conf=0.45, save=True, project="runs", name="preview", exist_ok=True)
for p in sorted(pathlib.Path("runs/preview").iterdir())[:4]:
    display(Show(str(p), width=340))

## 7. Make the phone file

`int8=True` stores the numbers more roughly — like writing 3.14 instead of
3.14159. The file gets about 4× smaller and runs about 4× faster on a phone,
and loses roughly 1–3% accuracy, which you cannot notice in practice.

In [ ]:
best = "runs/train/weights/best.pt"

YOLO(best).export(format="tflite", imgsz=640, int8=True, data="data.yaml")

import glob, shutil, pathlib, hashlib, json, datetime
tflite = sorted(glob.glob("runs/train/weights/**/*int8*.tflite", recursive=True) +
                glob.glob("runs/train/weights/**/*.tflite", recursive=True))[0]

pathlib.Path("export").mkdir(exist_ok=True)
out = pathlib.Path("export/model_int8.tflite")
shutil.copy(tflite, out)

manifest = {
    "model_file": "model_int8.tflite",
    "version": "1.0.0",
    "created": datetime.date.today().isoformat(),
    "input_size": 640,
    "precision": "int8",
    "classes": ["healthy", "unhealthy"],
    "sha256": hashlib.sha256(out.read_bytes()).hexdigest(),
    "trained_on": "PlantVillage (public) - single leaves on a plain background",
}
pathlib.Path("export/manifest.json").write_text(json.dumps(manifest, indent=2))

print(f"{out}  ->  {out.stat().st_size/1e6:.1f} MB")
print(open("export/manifest.json").read())

## 8. Check the phone file still works

Converting **never reports an error**, even when it broke something.
So compare the two files on the same photos before trusting it.

In [ ]:
import pathlib, random, numpy as np
from ultralytics import YOLO

original = YOLO("runs/train/weights/best.pt")
phone    = YOLO("export/model_int8.tflite")

photos = random.sample(sorted(pathlib.Path("data/images/val").iterdir()), 50)
agree = total = 0
for p in photos:
    a = original.predict(str(p), conf=0.45, verbose=False)[0]
    b = phone.predict(str(p),    conf=0.45, verbose=False)[0]
    ca = a.boxes.cls.cpu().numpy().astype(int).tolist() if a.boxes is not None else []
    cb = b.boxes.cls.cpu().numpy().astype(int).tolist() if b.boxes is not None else []
    total += 1
    agree += int(sorted(ca) == sorted(cb))

print(f"the two files agreed on {agree}/{total} photos ({agree/total:.0%})")
print("95% or better = the conversion is fine" if agree/total >= 0.95
      else "under 95% = re-export, something changed in conversion")

## 9. Download the two files

In [ ]:
from google.colab import files
files.download("export/model_int8.tflite")
files.download("export/manifest.json")

---

## Before you show this to anyone

This model was trained on **single leaves on a plain background**, in a lab.
It will look impressive on photos that look the same, and it will do badly on
a real canopy shot with dew, shadows and overlapping leaves.

That is not a bug. It is the point:

> Public data gets you a working demo in an afternoon.
> Your own field photos, labelled in Label Studio, are what make it work in the field.

Use this to learn the pipeline. Then replace the data with your own and run the
same steps again.